# Exercise 5: Real and synthetic code-switching

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Cross-Lingual Transfer · version 0.1.0*

> Evaluate real code-switched text separately from synthetic perturbations, with borrowing conventions and label-preservation checks.

**Setup.** `pip install sklearn-crfsuite scikit-learn sacrebleu torch transformers sentencepiece fasttext-wheel "stanza==1.11.0" requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Data come from commit-pinned GitHub repositories and are hash-checked: KazNERD and KyrgyzNER (Exercise 1), SIB-200 (Exercises 2 and 3), the X-WMT Kazakh–Kyrgyz test set (Exercise 4) and the UD 2.17 Turkish–German SAGT and Turkish IMST test files (Exercise 5). Sentence encoders and NLLB-200 are read from local copies (`MODELS_DIR`, `HF_HUB_DIR`, `NLLB_PATH`) or the Hub at the revisions pinned in the code; GlotLID from `GLOTLID_PATH`. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's helpers from `../code/remaining/transfer.py`. Embeddings, translations and model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

Stanza 1.11's Turkish model (trained on IMST) tags the gold words of the UD 2.17 Turkish–German SAGT test set; accuracy is split by SAGT's word-level `Lang` label. The synthetic condition replaces one Turkish noun per IMST test sentence with a German noun taken from SAGT, retaining an intended NOUN label; the resulting sentences still need contextual validation.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, sys
from pathlib import Path
import numpy as np, requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from transfer import SCHEMA, bio_spans, conversion_map, restore_span

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/xwmt/test/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
    UD.format("UD_Turkish_German-SAGT", "qtd_sagt-ud-test.conllu"): "0bbb06de66a490bfbb93d103a03d0d89e3835baf0af755a57b5fa5f885ae01bc",
    UD.format("UD_Turkish-IMST", "tr_imst-ud-test.conllu"): "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Script view (TurkicNLP transliterator); Turkish and English stay unchanged."""
    if lang in ("tur", "eng"):
        return text
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = Script.LATIN if lang in ("uzb", "aze") else Script.CYRILLIC
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def sib(lang):
    """All 1,004 SIB-200 rows of a language as (index_id, split, topic, text), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), split, r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return sorted(rows)

MODELS = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
          "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a"}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
NLLB_PATH = os.environ.get("NLLB_PATH", str(Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()))
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl", "uzn_Latn": "uzn_Latn",
             "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl", "eng_Latn": "eng_Latn", "fra_Latn": "fra_Latn", "azj_Latn": "azj_Latn"}
def model_source(name):
    rev = MODELS[name]
    for p in (Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser() / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              Path(os.environ.get("MODELS_DIR", CACHE / "models")) / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev
_ENC = {}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors, as in Meaning and Retrieval (same cache)."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            _ENC[model] = (AutoTokenizer.from_pretrained(NLLB_PATH), AutoModelForSeq2SeqLM.from_pretrained(NLLB_PATH).get_encoder().eval(), 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def macro_f1(gold, pred, labels):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=labels, average="macro", zero_division=0)

print("helpers loaded from code/remaining/transfer.py; schema map:", SCHEMA)

In [ ]:
import logging, stanza
logging.getLogger("stanza").setLevel(logging.ERROR)
def read(url):
    sents, words = [], []
    for line in fetch(url).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append(words)
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                misc = dict(x.split("=", 1) for x in c[9].split("|") if "=" in x)
                words.append({"form": c[1], "upos": c[3], "lang": misc.get("Lang", "?")})
    return sents
sagt = read(UD.format("UD_Turkish_German-SAGT", "qtd_sagt-ud-test.conllu"))
imst = read(UD.format("UD_Turkish-IMST", "tr_imst-ud-test.conllu"))
nlp = stanza.Pipeline("tr", processors="tokenize,pos", tokenize_pretokenized=True, verbose=False,
                      model_dir=os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser())), download_method=None)
def tag(sents):
    path = CACHE / ("ch16-stanza-" + hashlib.sha1(json.dumps([[w["form"] for w in s] for s in sents]).encode()).hexdigest()[:12] + ".json")
    if not path.exists():
        path.write_text(json.dumps([[w.upos for w in s.words] for s in nlp([[w["form"] for w in s] for s in sents]).sentences]))
    return json.loads(path.read_text())
# 1. Real code-switching: SAGT (Turkish-German conversation), tagged with Stanza's Turkish IMST model.
pred = tag(sagt)
acc = collections.defaultdict(lambda: [0, 0])
for s, p in zip(sagt, pred):
    for w, u in zip(s, p):
        if w["upos"] != "PUNCT":
            acc[w["lang"]][0] += u == w["upos"]; acc[w["lang"]][1] += 1
mixed = sum(len({w["lang"] for w in s if w["lang"] in ("tr", "de")}) == 2 for s in sagt)
print(f"SAGT test: {len(sagt)} sentences, {mixed} with both Turkish and German words; Stanza tr (IMST) UPOS accuracy by gold word language:")
conf = collections.defaultdict(collections.Counter)
for s, p in zip(sagt, pred):
    for w, u in zip(s, p):
        if w["upos"] != "PUNCT" and u != w["upos"]: conf[w["lang"]][(w["upos"], u)] += 1
for l, (k, n) in sorted(acc.items(), key=lambda x: -x[1][1]):
    print(f"  Lang={l:<6} {n:>6} words  {k / n:6.1%}   most frequent errors (gold -> predicted): {conf[l].most_common(3)}")
pred_de = collections.Counter(u for s, p in zip(sagt, pred) for w, u in zip(s, p) if w["lang"] == "de")
print("  tags the Turkish model gives to German words:", dict(pred_de.most_common(5)))
# 2. Synthetic perturbation: one Turkish noun per IMST test sentence replaced by a German noun from SAGT.
german_nouns = sorted({w["form"] for s in sagt for w in s if w["lang"] == "de" and w["upos"] == "NOUN" and w["form"].isalpha()})
rng = random.Random(0); synth, where = [], []
for s in imst:
    nouns = [i for i, w in enumerate(s) if w["upos"] == "NOUN" and w["form"].isalpha()]
    if not nouns: continue
    i = rng.choice(nouns); t = [dict(w) for w in s]; t[i] = {"form": rng.choice(german_nouns), "upos": "NOUN", "lang": "de"}
    synth.append(t); where.append(i)
base = tag([imst_s for imst_s in imst if any(w["upos"] == "NOUN" and w["form"].isalpha() for w in imst_s)])
sp = tag(synth)
orig = [s for s in imst if any(w["upos"] == "NOUN" and w["form"].isalpha() for w in s)]
k_ins = sum(p[i] == "NOUN" for p, i in zip(sp, where))
k_orig = sum(p[i] == "NOUN" for p, i in zip(base, where))
rest = lambda P, S: sum(u == w["upos"] for p, s, i in zip(P, S, where) for j, (u, w) in enumerate(zip(p, s)) if j != i and w["upos"] != "PUNCT")
n_rest = sum(1 for s, i in zip(orig, where) for j, w in enumerate(s) if j != i and w["upos"] != "PUNCT")
print(f"\nIMST test with one German noun inserted per sentence ({len(synth)} sentences, {len(german_nouns)} distinct German nouns from SAGT):")
print(f"  inserted German noun tagged NOUN: {k_ins / len(where):.1%}  (the original Turkish noun: {k_orig / len(where):.1%})")
print(f"  other words: {rest(base, orig) / n_rest:.1%} correct before, {rest(sp, synth) / n_rest:.1%} after the insertion")
de_nouns = [(w["form"], u) for s, p in zip(sagt, pred) for w, u in zip(s, p) if w["lang"] == "de" and w["upos"] == "NOUN"]
print(f"  for comparison, German nouns in real SAGT sentences tagged NOUN: {sum(u == 'NOUN' for _, u in de_nouns) / len(de_nouns):.1%} of {len(de_nouns)}")

## Worked solution

Output of `experiments/ch16_exercises/run_ch16_exercises.py`:

```
SAGT test: 805 sentences, 762 with both Turkish and German words; Stanza tr (IMST) UPOS accuracy by gold word language:
  Lang=de       7141 words    2.8%   most frequent errors (gold -> predicted): [(('ADV', 'PROPN'), 797), (('NOUN', 'PROPN'), 794), (('ADV', 'NOUN'), 611)]
  Lang=tr       5339 words   74.6%   most frequent errors (gold -> predicted): [(('ADV', 'CCONJ'), 249), (('ADV', 'ADJ'), 137), (('INTJ', 'NOUN'), 75)]
  Lang=qtd       182 words   33.5%   most frequent errors (gold -> predicted): [(('NOUN', 'PROPN'), 116), (('ADJ', 'PROPN'), 2), (('PROPN', 'NOUN'), 2)]
  Lang=en         41 words   58.5%   most frequent errors (gold -> predicted): [(('NOUN', 'PROPN'), 10), (('ADP', 'INTJ'), 2), (('ADJ', 'PROPN'), 1)]
  Lang=es          1 words  100.0%   most frequent errors (gold -> predicted): []
  Lang=fr          1 words  100.0%   most frequent errors (gold -> predicted): []
  tags the Turkish model gives to German words: {'PROPN': 4271, 'NOUN': 2393, 'INTJ': 200, 'AUX': 65, 'CCONJ': 57}

IMST test with one German noun inserted per sentence (861 sentences, 493 distinct German nouns from SAGT):
  inserted German noun tagged NOUN: 12.7%  (the original Turkish noun: 96.1%)
  other words: 93.8% correct before, 93.0% after the insertion
  for comparison, German nouns in real SAGT sentences tagged NOUN: 2.7% of 819
```

**Real and synthetic mixed text give different results.** On SAGT, Stanza's Turkish model has UPOS accuracy of 74.6% on Turkish-labelled words and 2.8% on German-labelled words. It predicts PROPN for 4,271 German-labelled words. Capitalisation is a possible influence, but this run does not test that explanation.

In the IMST perturbation, 12.7% of inserted German nouns receive NOUN, compared with 2.7% of German nouns in SAGT. Accuracy on the other IMST words changes from 93.8% to 93.0%. These samples differ in domain, context and annotation; their gap cannot be attributed solely to code-switching.

Replacing a noun with an arbitrary German noun retains an intended NOUN label, but does not guarantee a natural or grammatical sentence. Fluent review is needed to validate the perturbation. SAGT's language labels define the groups scored here; established loans and ambiguous forms must be interpreted under its guidelines.

## Your turn — extension

Repeat with the Turkish–English BUTR treebank, and build a better perturbation: insert whole German noun phrases from SAGT (with their determiners) and compare the error profile with real SAGT switches.